# 校準分析（Calibration Analysis）

載入回測輸出（`reports/backtest_results.csv`），檢視：
1. 三種合併策略的校準曲線（Reliability Diagram，規格 6.2）
2. 各 Agent 精度隨回測進行的演化（loose → tight coupling）

先執行 `python backtest/run_backtest.py --mode synthetic`（或 `--mode finmind`）產生資料。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from backtest.metrics import calibration_bins, ece, mean_brier, directional_accuracy

df = pd.read_csv(ROOT / 'reports' / 'backtest_results.csv')
print(f'{len(df)} predictions, {df.ticker.nunique()} tickers, {df.as_of_date.min()} ~ {df.as_of_date.max()}')
df.head()

In [ ]:
STRATEGIES = {
    'baseline_a': 'Baseline A: single agent',
    'baseline_b': 'Baseline B: simple average',
    'precision_weighted': 'Precision-weighted',
}
outcomes = df['outcome'].astype(int).tolist()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (key, label) in zip(axes, STRATEGIES.items()):
    ps = df[f'{key}_p'].tolist()
    bins = calibration_bins(ps, outcomes)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.plot([b[0] for b in bins], [b[1] for b in bins], 'o-')
    ax.set_title(f'{label}\nBrier={mean_brier(ps, outcomes):.4f}, ECE={ece(ps, outcomes):.4f}')
    ax.set_xlabel('Predicted P(bullish)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Observed hit rate')
plt.tight_layout()

In [ ]:
# 各 Agent 的有效精度演化：冷啟動時均等（loose coupling），
# 隨精度記錄累積，權重逐漸向可靠 Agent 集中（tight coupling）
prec_cols = [c for c in df.columns if c.startswith('precision_') and c != 'precision_weighted_signal' and c != 'precision_weighted_p']
prec_cols = [c for c in prec_cols if df[c].dtype != object]

fig, ax = plt.subplots(figsize=(10, 4))
for ticker, g in df.groupby('ticker'):
    for c in prec_cols:
        ax.plot(pd.to_datetime(g['as_of_date']), g[c], alpha=0.4,
                label=c.replace('precision_', '') if ticker == df['ticker'].iloc[0] else None,
                color='C0' if 'fundamentals' in c else 'C1')
ax.set_ylabel('Effective precision')
ax.set_title('Precision evolution: loose coupling -> tight coupling')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# 彙總表（規格 6.3 三指標）
rows = []
for key, label in STRATEGIES.items():
    ps = df[f'{key}_p'].tolist()
    sigs = df[f'{key}_signal'].tolist()
    acc, n_dir = directional_accuracy(sigs, outcomes)
    rows.append({'strategy': label, 'mean_brier': mean_brier(ps, outcomes),
                 'directional_accuracy': acc, 'ece': ece(ps, outcomes), 'n_directional': n_dir})
pd.DataFrame(rows).round(4)